In [1]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-08-12 00:49:16--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-08-12 00:49:16 (29.7 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [9]:
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
from openai import OpenAI

from sentence_transformers import SentenceTransformer
from minsearch import VectorSearch
from tqdm import tqdm

import numpy as np

In [10]:
load_dotenv()
openai_client = OpenAI()

In [11]:
documents = load_faq_data()
index = build_index(documents)

In [12]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [14]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
len(vectors)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [01:42<00:00,  3.54s/it]


1406

In [15]:
X = np.array(vectors)

In [16]:
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [28]:
from rag_helper import RAGBase

class RAGVector(RAGBase):
    def __init__(self, 
                 embedder,
                 **kwargs
                ):
        super().__init__(
            **kwargs
            )
        
        self.embedder = embedder
        
    def search(self, query, num_results=5):
        query_as_vector = self.embedder.encode(query)
        
        return self.index.search(
            query_as_vector,
            filter_dict={'course': self.course},
            num_results=num_results
        )
        
assistant = RAGVector(
    embedder = model,
    index=vindex,
    llm_client=openai_client
)
query = "I just found out about the program, can I still sign up?"
query = "the program has already begun, can I still sign up?"
assistant.rag(query)

'Yes — you can still sign up and join even though the program has already begun. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [29]:
assistant.course

'llm-zoomcamp'